# Python AST 快速入门

NineToothed 的 `generation.py` 核心是 AST 变换。这个 notebook 帮你掌握看懂它所需的 AST 知识。


## 1. AST 是什么？

AST = Abstract Syntax Tree。Python 执行代码前，先把源码解析成一棵树。

```
"output = lhs + rhs"
         |
     Module
       +-- Assign
            +-- targets: [Name('output')]
            +-- value: BinOp
                         +-- left: Name('lhs')
                         +-- op: Add()
                         +-- right: Name('rhs')
```

NineToothed: 遍历这棵树, 把 Python 节点替换成 Triton 节点。


In [30]:
import ast

source = "output = lhs + rhs"
tree = ast.parse(source)

print("type(tree):", type(tree).__name__)
print()
print(ast.dump(tree, indent=2))


type(tree): Module

Module(
  body=[
    Assign(
      targets=[
        Name(id='output', ctx=Store())],
      value=BinOp(
        left=Name(id='lhs', ctx=Load()),
        op=Add(),
        right=Name(id='rhs', ctx=Load())))],
  type_ignores=[])


## 2. 核心三件套

- `ast.parse` -- 源码 -> AST
- `ast.dump` -- AST -> 可读文本 (调试用)
- `ast.unparse` -- AST -> 源码 (代码生成的最后一步)


In [31]:
tree = ast.parse("x = 1 + 2")
print("=== ast.dump (树结构) ===")
print(ast.dump(tree, indent=2))

print("\n=== ast.unparse (还原成源码) ===")
print(ast.unparse(tree))


=== ast.dump (树结构) ===
Module(
  body=[
    Assign(
      targets=[
        Name(id='x', ctx=Store())],
      value=BinOp(
        left=Constant(value=1),
        op=Add(),
        right=Constant(value=2)))],
  type_ignores=[])

=== ast.unparse (还原成源码) ===
x = 1 + 2


### 看 NineToothed 的用户代码解析成什么样


In [32]:
src = (
    "def add_kernel(\n"
    "    lhs: Tensor(1).tile((BLOCK_SIZE,)),\n"
    "    rhs: Tensor(1).tile((BLOCK_SIZE,)),\n"
    "    output: Tensor(1).tile((BLOCK_SIZE,)),\n"
    "):\n"
    "    output = lhs + rhs\n"
)

tree = ast.parse(src)
func_def = tree.body[0]
print("函数名:", func_def.name)
print("参数个数:", len(func_def.args.args))
print("函数体语句数:", len(func_def.body))
print()

assign = func_def.body[0]
print("赋值语句 dump:")
print(ast.dump(assign, indent=2))


函数名: add_kernel
参数个数: 3
函数体语句数: 1

赋值语句 dump:
Assign(
  targets=[
    Name(id='output', ctx=Store())],
  value=BinOp(
    left=Name(id='lhs', ctx=Load()),
    op=Add(),
    right=Name(id='rhs', ctx=Load())))


## 3. 常见节点类型

| 节点 | 对应语法 | 关键属性 |
|------|---------|---------|
| `Module` | 整个文件 | `.body` |
| `FunctionDef` | `def f():` | `.name`, `.args`, `.body` |
| `Assign` | `x = y` | `.targets[0]`, `.value` |
| `BinOp` | `a + b` | `.left`, `.op`, `.right` |
| `Call` | `f(a, b)` | `.func`, `.args` |
| `Name` | `x` | `.id` |
| `Constant` | `42` | `.value` |
| `Subscript` | `a[i]` | `.value`, `.slice` |
| `Attribute` | `a.b` | `.value`, `.attr` |


In [33]:
def show(source_label, source_code):
    print(f"\n=== {source_label} ===")
    tree = ast.parse(source_code)
    node = tree.body[0]
    print(f"类型: {type(node).__name__}")
    if hasattr(node, 'value'):
        v = node.value
        print(f"  value: {type(v).__name__}")
        if isinstance(v, ast.BinOp):
            print(f"    left: {ast.dump(v.left)}")
            print(f"    op:   {type(v.op).__name__}")
            print(f"    right:{ast.dump(v.right)}")
        elif isinstance(v, ast.Call):
            print(f"    func: {ast.dump(v.func)}")
            print(f"    args: {[ast.dump(a) for a in v.args]}")

show("赋值", "x = a + b")
show("函数调用", "y = f(x, 42)")
show("下标", "z = arr[i]")
show("属性", "w = obj.attr")



=== 赋值 ===
类型: Assign
  value: BinOp
    left: Name(id='a', ctx=Load())
    op:   Add
    right:Name(id='b', ctx=Load())

=== 函数调用 ===
类型: Assign
  value: Call
    func: Name(id='f', ctx=Load())
    args: ["Name(id='x', ctx=Load())", 'Constant(value=42)']

=== 下标 ===
类型: Assign
  value: Subscript

=== 属性 ===
类型: Assign
  value: Attribute


## 4. NodeTransformer: 遍历和替换

继承 `ast.NodeTransformer`, 重写 `visit_*` 方法, 返回新节点替换旧节点。


In [34]:
class AddOne(ast.NodeTransformer):
    def visit_Constant(self, node):
        if isinstance(node.value, int):
            return ast.Constant(value=node.value + 1)
        return node

source = "x = 1 + 2"
tree = ast.parse(source)
print("原始:", ast.unparse(tree))
AddOne().visit(tree)
print("变换后:", ast.unparse(tree))


原始: x = 1 + 2
变换后: x = 2 + 3


In [35]:
class ReplaceAdd(ast.NodeTransformer):
    def visit_BinOp(self, node):
        self.generic_visit(node)
        if (isinstance(node.op, ast.Add)
            and ast.dump(node.left) == ast.dump(node.right)):
            return ast.BinOp(
                left=ast.Constant(value=2),
                op=ast.Mult(),
                right=node.left
            )
        return node

source = "z = x + x"
tree = ast.parse(source)
print("原始:", ast.unparse(tree))
ReplaceAdd().visit(tree)
print("变换后:", ast.unparse(tree))


原始: z = x + x
变换后: z = 2 * x


### 模拟：把 Name 替换成 tl.load 调用

`generation.py` 的 `visit_Call` 检测 tensor 操作, 替换为 `tl.load`。


In [36]:
class MockCodeGen(ast.NodeTransformer):
    def visit_Name(self, node):
        if node.id in ('lhs', 'rhs'):
            return ast.Call(
                func=ast.Attribute(
                    value=ast.Name(id='tl', ctx=ast.Load()),
                    attr='load', ctx=ast.Load()
                ),
                args=[node], keywords=[]
            )
        return node

source = "output = lhs + rhs"
tree = ast.parse(source)
print("原始:", ast.unparse(tree))
MockCodeGen().visit(tree)
print("变换后:", ast.unparse(tree))


原始: output = lhs + rhs
变换后: output = tl.load(lhs) + tl.load(rhs)


## 6. generation.py 的两种 visit 模式

**模式 A**: 直接改 AST 结构
```python
def visit_FunctionDef(self, node):
    node.decorator_list = [Symbol("triton.jit").node]
    return node
```

**模式 B**: 检测 tensor 操作, 调用 _generate_*
```python
def visit_Subscript(self, node):
    if self._in_context(node.value):
        return self._generate_load(tensor, ...)
```

`_generate_*` 返回 AST 节点, `visit_*` 用它们替换原始节点。


## 5. 自动分发机制：为什么不需要手动调用 visit_*？

`ast.NodeTransformer` 的基类自带自动分发逻辑。只需要调用 `self.visit(tree)` 一句，它自动递归遍历整个树：

```python
class NodeTransformer:
    def visit(self, node):
        method = 'visit_' + node.__class__.__name__
        visitor = getattr(self, method, self.generic_visit)
        return visitor(node)

    def generic_visit(self, node):
        for field, value in ast.iter_fields(node):
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, AST):
                        self.visit(item)
            elif isinstance(value, AST):
                self.visit(value)
        return node
```

没有对应的 `visit_*` 方法就走 `generic_visit`，继续往下递归。所以只需要定义想处理的节点类型。

### 5.1 追踪 dispatch 过程

In [37]:
import ast

class TraceTransformer(ast.NodeTransformer):
    def visit_Module(self, node):
        print('  -> visit_Module')
        self.generic_visit(node)
        print('  <- visit_Module (done)')
        return node

    def visit_FunctionDef(self, node):
        print(f'  -> visit_FunctionDef (name={node.name})')
        self.generic_visit(node)
        print('  <- visit_FunctionDef (done)')
        return node

    def visit_Assign(self, node):
        print('  -> visit_Assign')
        self.generic_visit(node)
        print('  <- visit_Assign (done)')
        return node

    def visit_Name(self, node):
        print(f"    visit_Name(id='{node.id}')")
        return node

    def visit_Constant(self, node):
        print(f'    visit_Constant(value={node.value})')
        return node

source = '''
def add_kernel(lhs, rhs, output):
    output = lhs + rhs
'''

print('原始源码:')
print(source.strip())
print('\nDispatch 追踪:')
tree = ast.parse(source)
TraceTransformer().visit(tree)

原始源码:
def add_kernel(lhs, rhs, output):
    output = lhs + rhs

Dispatch 追踪:
  -> visit_Module
  -> visit_FunctionDef (name=add_kernel)
  -> visit_Assign
    visit_Name(id='output')
    visit_Name(id='lhs')
    visit_Name(id='rhs')
  <- visit_Assign (done)
  <- visit_FunctionDef (done)
  <- visit_Module (done)


### 5.2 generation.py 的真实 dispatch 树

`CodeGenerator` 只定义了这些 `visit_*` 方法，**没有定义** `visit_BinOp`、`visit_Add` 等：

| 定义了的 | 没定义的（走 generic_visit） |
|---------|---------------------------|
| `visit_Module` | `visit_BinOp` |
| `visit_FunctionDef` | `visit_Add`, `visit_Mult` |
| `visit_Call` | `visit_Sub` |
| `visit_Subscript` | |
| `visit_Attribute` | |
| `visit_Name` | |
| `visit_Assign` | |

没定义 `visit_BinOp` 意味着 `lhs + rhs` 的 `BinOp` 节点不会被替换。但它的子节点 `Name('lhs')` 和 `Name('rhs')` 会被 `visit_Name` 捕获，替换成 `tl.load(...)`。

最终：`tl.load(lhs_ptr, ...) + tl.load(rhs_ptr, ...)`

### 5.3 generation.py 的真实 visit_* 代码

In [38]:
print('=' * 70)
print('visit_Name  -- generation.py:320-326')
print('=' * 70)
print('''
def visit_Name(self, node):
    self.generic_visit(node)
    if self._in_context(node) and isinstance(node.ctx, ast.Load):
        return self._generate_load(self._context[node.id])
    return node
''')

print('=' * 70)
print('visit_Assign  -- generation.py:328-337')
print('=' * 70)
print('''
def visit_Assign(self, node):
    if len(node.targets) == 1:
        target = node.targets[0]
        if self._in_context(target):
            self.generic_visit(node)
            return ast.Expr(
                self._generate_store(self._context[target.id], node.value)
            )
    ...
''')

print('=' * 70)
print('visit_Subscript  -- generation.py:268-275')
print('=' * 70)
print('''
def visit_Subscript(self, node):
    def _generate_load():
        return self._generate_load(
            tensor,
            indices=node.slice.elts if isinstance(node.slice, ast.Tuple)
                     else (node.slice,),
        )
    if self._in_context(node.value) and isinstance(...):
        return _generate_load()
    ...
''')

print('=' * 70)
print('visit_FunctionDef  -- generation.py:151-204 (摘要)')
print('=' * 70)
print('''
def visit_FunctionDef(self, node):
    self._func_def = node
    self._invariants = {}
    self.generic_visit(node)  # <<< 触发子节点遍历
    # ... 提取符号,重写参数列表
    self._func_def.decorator_list = [Symbol("triton.jit").node]
    self._launch = self._generate_launch(...)
    return node
''')

visit_Name  -- generation.py:320-326

def visit_Name(self, node):
    self.generic_visit(node)
    if self._in_context(node) and isinstance(node.ctx, ast.Load):
        return self._generate_load(self._context[node.id])
    return node

visit_Assign  -- generation.py:328-337

def visit_Assign(self, node):
    if len(node.targets) == 1:
        target = node.targets[0]
        if self._in_context(target):
            self.generic_visit(node)
            return ast.Expr(
                self._generate_store(self._context[target.id], node.value)
            )
    ...

visit_Subscript  -- generation.py:268-275

def visit_Subscript(self, node):
    def _generate_load():
        return self._generate_load(
            tensor,
            indices=node.slice.elts if isinstance(node.slice, ast.Tuple)
                     else (node.slice,),
        )
    if self._in_context(node.value) and isinstance(...):
        return _generate_load()
    ...

visit_FunctionDef  -- generation.py:151-204 (摘要)

### 5.4 核心执行流程

完整 dispatch 链：

```
self.visit(tree)  <- Module
  -> visit_Module(node)
       -> generic_visit -> 遍历 body -> FunctionDef
            -> visit_FunctionDef(node)
                 -> self.generic_visit(node) -> 遍历 body
                      -> visit_Assign(node)  <- "output = lhs + rhs"
                           -> self.generic_visit -> 遍历 value
                                -> (BinOp: 没有 visit_BinOp -> generic_visit)
                                     -> visit_Name('lhs') -> _generate_load
                                     -> visit_Name('rhs') -> _generate_load
                           -> return Expr(tl.store(...))

ast.unparse(tree) -> "tl.store(out, tl.load(lhs) + tl.load(rhs), mask=...)"
```

In [39]:
print("""总结
====
1. ast.parse + ast.unparse = 源码 <-> AST 双向转换
2. ast.NodeTransformer.visit_* = 匹配节点 -> 替换
3. CodeGenerator._generate_* 返回 AST 节点
""")

总结
====
1. ast.parse + ast.unparse = 源码 <-> AST 双向转换
2. ast.NodeTransformer.visit_* = 匹配节点 -> 替换
3. CodeGenerator._generate_* 返回 AST 节点

